# 🎧 AudiobookMaker on Colab: Try Engines

Hear the same text read by several TTS engines in your narrator's voice, then pick one for the book.

1. Installs the app and each engine you list, every engine in an environment of its own (they need different library versions; this way they all stay installed).
2. Reads your test text with your narrator clip in each engine, one after the other, freeing the GPU in between.
3. Shows a table (speed, load time, GPU memory, and word errors found by Whisper) and an audio player per engine.

Then open **AudiobookMaker_Colab.ipynb** ("Make the audiobook"), install only the engine you chose, and make the book.

> ⚠️ The engine notes in this project (best uses, speeds, quality) come from **automated test runs by an AI coding assistant (Claude)**, scored mostly by Whisper rather than by listening. Your own ears, on your own voice and text, decide.

| Engine | Voice | Languages | Licence |
|---|---|---|---|
| `qwen` | clone, describe, built-in readers | 10 | Apache-2.0 |
| `higgs` | clone | 102 (best for Hindi) | non-commercial, Creator grant |
| `omnivoice` | clone, describe (tags) | 600+ | non-commercial |
| `moss` | clone | 31 | Apache-2.0 |
| `indextts` | clone (no transcript) | 5 | bilibili licence |
| `fish` | clone (transcript) | 83 | non-commercial |
| `breeze` | clone (transcript), style instruction | English, Chinese | non-commercial |
| `ming` | clone (transcript) | 7 | MIT |
| `f5tts` | clone | English, Chinese | non-commercial |

Downloads per engine are large (MOSS ~13 GB, Fish ~11 GB, Higgs ~9 GB); `DELETE_MODELS_AFTER` frees the disk after each one.

### ⚡ GPU check

In [ ]:
# Check if GPU accelerator is assigned
import subprocess
try:
    gpu_info = subprocess.check_output("nvidia-smi").decode("utf-8")
    print(gpu_info)
    print("✅ GPU Accelerator is correctly assigned!")
except FileNotFoundError:
    print("\n" + "="*80)
    print("⚠️ WARNING: GPU accelerator not detected (nvidia-smi not found)!")
    print("The application will run, but text-to-speech generation will be EXTREMELY slow.")
    print("To enable GPU:")
    print("  1. Go to top menu: Runtime -> Change runtime type.")
    print("  2. Select 'T4 GPU' under Hardware accelerator.")
    print("="*80 + "\n")

### 📂 Get the app

In [ ]:
import os

# Google Colab working directory is /content
COLAB_WORKING = '/content'
os.chdir(COLAB_WORKING)

# Check if we are already in a repository root
if os.path.exists("audiobook_rust") and os.path.exists("requirements.txt"):
    print("Already in the AudiobookMaker repository root directory:", os.getcwd())
elif os.path.exists("AudiobookMaker/audiobook_rust"):
    print("Found AudiobookMaker directory. Changing directory...")
    os.chdir("AudiobookMaker")
    print("Current directory:", os.getcwd())
else:
    print("Cloning AudiobookMaker repository...")
    clone_status = os.system("git clone https://github.com/MSpider3/AudiobookMaker.git")
    if clone_status != 0:
        raise RuntimeError("Failed to clone the repository!")
    
    if os.path.exists("AudiobookMaker"):
        os.chdir("AudiobookMaker")
        print("Successfully cloned and entered AudiobookMaker directory.")
    else:
        raise FileNotFoundError("Could not find the cloned AudiobookMaker directory!")

# Final check
if not os.path.exists("audiobook_rust"):
    raise FileNotFoundError("Invalid workspace: 'audiobook_rust' directory not found at " + os.getcwd())

### 📦 Install the app's requirements

In [ ]:
import os
import sys

print("Updating system packages and installing FFmpeg, SoX & OpenSSH...")
os.system("apt-get update && apt-get install -y ffmpeg sox libsox-fmt-all openssh-client")

print("\nInstalling Python dependencies (this will take a few minutes)...")
pip_status = os.system(f"{sys.executable} -m pip install -r requirements.txt")
if pip_status != 0:
    print("Warning: pip install -r requirements.txt returned a non-zero status.")

print("\nInstalling maturin (needed to build python-rust PyO3 modules)...")
os.system(f"{sys.executable} -m pip install maturin")


### 🎛️ Choose the engines, your voice and a test text

In [ ]:
# ── What to try ──────────────────────────────────────────────────────────────
ENGINES = ["qwen", "higgs", "omnivoice"]   # any of: qwen, higgs, omnivoice, moss, indextts, fish, breeze, ming, f5tts
VOICE_FILE = ""         # narrator clip, 5-30 s (.wav): a path or a Google Drive / HTTP link
VOICE_TRANSCRIPT = ""   # the exact words spoken in the clip (fish, breeze and ming need it; the others read it better with it)
LANGUAGE = "English"    # language of the text, e.g. "English", "Hindi", "Chinese"
TEST_TEXT = """The village sat at the end of a long, crooked road, and beyond it there was nothing but grey sea.
That changed on a wet Tuesday in early October, when a man in a yellow raincoat nailed a notice to the door of the village hall."""
DELETE_MODELS_AFTER = True   # delete each engine's download after its turn: the disk here is small
CHECK_WITH_WHISPER = True    # also transcribe each result with Whisper and count word errors

### 📦 Install the engines (each in its own environment)

In [ ]:
import os
import subprocess
import sys

def resolve_file(path_or_url, default_name="narrator_voice.wav"):
    if not path_or_url or os.path.exists(path_or_url):
        return path_or_url
    out = os.path.abspath(default_name)
    if "drive.google.com" in path_or_url:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        return os.path.abspath(gdown.download(path_or_url, out, quiet=False, fuzzy=True))
    subprocess.run(["wget", "-q", "-O", out, path_or_url], check=True)
    return out

voice_path = resolve_file(VOICE_FILE)
if not voice_path or not os.path.exists(voice_path):
    raise FileNotFoundError("Set VOICE_FILE above to your narrator clip.")

for engine in ENGINES:
    if engine == "qwen":
        continue
    print(f"Installing {engine} in an environment of its own...")
    if os.system(f"{sys.executable} -m audiobook_factory.engine_envs install {engine}") != 0:
        print(f"⚠️ {engine} did not install - it will be reported as failed below.")
os.system(f"{sys.executable} -m audiobook_factory.engine_envs list")

### ▶️ Hear them
Each engine loads, reads the text and is freed before the next one. Slow engines take a few minutes.

In [ ]:
from IPython.display import Audio, Markdown, display
from audiobook_factory import engine_trial

results = engine_trial.try_engines(ENGINES, voice_path, VOICE_TRANSCRIPT, TEST_TEXT, "engine_trial",
                                   language=LANGUAGE, delete_models=DELETE_MODELS_AFTER)
if CHECK_WITH_WHISPER:
    engine_trial.score_with_whisper(results, TEST_TEXT, LANGUAGE)
display(Markdown(engine_trial.table(results)))
for result in results:
    if result.ok:
        display(Markdown(f"**{result.engine}**" + (f" — Whisper heard: _{result.heard}_" if result.heard else "")))
        display(Audio(result.wav_path))

### ✅ Next
Open **AudiobookMaker_Colab.ipynb**, set `ENGINE` in its engine cell to the one you liked, and make the audiobook.